# Core Dashboard Readiness Contract v1

本实验验证 dashboard readiness overview 已沉到 `src/core`，API、dashboard 和 schedule 可以复用同一个 gate 契约，避免各 surface 单独复制规则。

In [1]:
from pathlib import Path
import json
import sys

current = Path.cwd().resolve()
repo_root = next(candidate for candidate in (current, *current.parents) if (candidate / "src").is_dir())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.core import build_dashboard_readiness_overview

OUTPUT_DIR = repo_root / "research/core_dashboard_readiness_contract_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 输入样本

样本覆盖一个 review 状态和一个 blocked 状态。review 状态要求 manifest、production health、notebook card、requested model、model run 五类 gate 同时出现。

In [2]:
review_payload = {
    "manifest_source": {
        "kind": "analysis_manifest",
        "path": "matrix_card/analysis_manifest.json",
        "schema_version": "artifact.v1",
        "manifest_hash": "a" * 64,
        "producer": "src.analysis.exports",
    },
    "production_health": {
        "schema_version": "schedule.production_health.v1",
        "health_status": "healthy",
        "is_healthy": True,
        "missing_required_paths": [],
    },
    "notebook_card_readiness": {
        "schema_version": "dashboard.notebook_card_readiness.v1",
        "summary": {
            "production_published_count": 1,
            "production_template_count": 1,
            "production_manifest_missing_count": 0,
            "production_registry_mismatch_count": 0,
        },
    },
    "requested_model_contract": {
        "schema_version": "model.requested_contract_health.v1",
        "requested_model_count": 32,
        "completed_requested_model_count": 31,
        "missing_from_ledger": ["sthgcn_mse"],
        "incomplete_requested_models": ["sthgcn_mse"],
        "all_requested_models_completed": False,
    },
    "model_run_coverage": {
        "schema_version": "model.run_ledger_summary.v1",
        "expected_model_count": 2,
        "trained_model_count": 2,
        "backtested_model_count": 2,
        "all_models_completed": True,
    },
}

blocked_payload = {"top_models": [{"model": "ridge_mse"}]}

review_overview = build_dashboard_readiness_overview(review_payload)
blocked_overview = build_dashboard_readiness_overview(blocked_payload)

review_overview, blocked_overview

({'schema_version': 'dashboard.readiness_overview.v1',
  'status': 'review',
  'gate_count': 5,
  'pass_count': 3,
  'review_count': 2,
  'block_count': 0,
  'gates': [{'gate': 'dashboard_manifest',
    'label': 'Dashboard Manifest',
    'status': 'pass',
    'detail': 'analysis_manifest / complete hash aaaaaaaaaa',
    'source_schema_version': 'artifact.v1'},
   {'gate': 'production_health',
    'label': 'Production Health',
    'status': 'pass',
    'detail': 'healthy / missing 0',
    'source_schema_version': 'schedule.production_health.v1'},
   {'gate': 'notebook_cards',
    'label': 'Notebook Cards',
    'status': 'review',
    'detail': 'published 1 / template 1 / missing 0 / mismatch 0',
    'source_schema_version': 'dashboard.notebook_card_readiness.v1'},
   {'gate': 'requested_models',
    'label': 'Requested Models',
    'status': 'review',
    'detail': '31 / 32 complete / missing 1 / incomplete 1',
    'source_schema_version': 'model.requested_contract_health.v1'},
   {'gat

## 契约断言

断言 gate 顺序、状态计数和关键 detail。后续如果 dashboard/API/schedule 需要扩展 gate，应先改 core 契约和这里的证据。

In [3]:
assert review_overview["schema_version"] == "dashboard.readiness_overview.v1"
assert review_overview["status"] == "review"
assert review_overview["gate_count"] == 5
assert review_overview["pass_count"] == 3
assert review_overview["review_count"] == 2
assert review_overview["block_count"] == 0
assert [gate["gate"] for gate in review_overview["gates"]] == [
    "dashboard_manifest",
    "production_health",
    "notebook_cards",
    "requested_models",
    "model_runs",
]

review_gates = {gate["gate"]: gate for gate in review_overview["gates"]}
assert review_gates["dashboard_manifest"]["detail"] == "analysis_manifest / complete hash aaaaaaaaaa"
assert review_gates["production_health"]["detail"] == "healthy / missing 0"
assert review_gates["notebook_cards"]["status"] == "review"
assert review_gates["requested_models"]["detail"] == "31 / 32 complete / missing 1 / incomplete 1"
assert review_gates["model_runs"]["status"] == "pass"

assert blocked_overview["status"] == "blocked"
assert blocked_overview["gate_count"] == 1
assert blocked_overview["block_count"] == 1
assert blocked_overview["gates"][0]["gate"] == "dashboard_manifest"
assert blocked_overview["gates"][0]["detail"] == "missing / missing hash "

In [4]:
analysis_summary = {
    "schema_version": "research.core_dashboard_readiness_contract_v1.summary.v1",
    "experiment": "core_dashboard_readiness_contract_v1",
    "core_contract": "dashboard.readiness_overview.v1",
    "review_status": review_overview["status"],
    "review_gate_count": review_overview["gate_count"],
    "review_pass_count": review_overview["pass_count"],
    "review_review_count": review_overview["review_count"],
    "review_block_count": review_overview["block_count"],
    "review_gate_order": [gate["gate"] for gate in review_overview["gates"]],
    "blocked_status": blocked_overview["status"],
    "blocked_gate_count": blocked_overview["gate_count"],
    "blocked_block_count": blocked_overview["block_count"],
    "promotion_decision": "promote_to_src_core_contract",
    "risk_note": "当前样本覆盖状态聚合和 gate 顺序，不替代真实模型训练回测。",
}

summary_path = OUTPUT_DIR / "analysis_summary.json"
summary_path.write_text(json.dumps(analysis_summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
analysis_summary

{'schema_version': 'research.core_dashboard_readiness_contract_v1.summary.v1',
 'experiment': 'core_dashboard_readiness_contract_v1',
 'core_contract': 'dashboard.readiness_overview.v1',
 'review_status': 'review',
 'review_gate_count': 5,
 'review_pass_count': 3,
 'review_review_count': 2,
 'review_block_count': 0,
 'review_gate_order': ['dashboard_manifest',
  'production_health',
  'notebook_cards',
  'requested_models',
  'model_runs'],
 'blocked_status': 'blocked',
 'blocked_gate_count': 1,
 'blocked_block_count': 1,
 'promotion_decision': 'promote_to_src_core_contract',
 'risk_note': '当前样本覆盖状态聚合和 gate 顺序，不替代真实模型训练回测。'}

## 中文结论

- 做了什么：验证 `dashboard.readiness_overview.v1` 已作为 `src/core` 契约承载 dashboard manifest、production health、notebook card、requested model、model run 五类 gate，API/dashboard/schedule 可复用同一规则。
- 核心指标：review 样本 `gate_count=5`、`pass_count=3`、`review_count=2`、`block_count=0`；blocked 样本 `gate_count=1`、`block_count=1`。
- regime 差异：本次不是市场行情 regime 回测，而是平台 readiness regime；完整 manifest 且局部未完成时进入 review，缺失 manifest 时进入 blocked。
- 是否晋升：晋升到 `src/core`，因为这是跨 API、dashboard、schedule 的稳定平台契约，不应散落在页面或执行入口。
- 风险：当前只验证 gate 聚合和状态解释，不代表所有 QuantBench 模型已经完成训练回测。
- 下一步：新增 gate 或改变晋升规则时，先扩展 core 契约、测试和本 notebook，再让 dashboard/API/schedule 读取统一输出。